# HelmNet: Helmet vs No-Helmet Image Classification

This completed notebook follows the same modeling flow used in the **Monkey Species Classification** notebook:

1. Load and inspect image data
2. Explore sample images and class balance
3. Preprocess images
4. Split into train, validation, and test sets
5. Build multiple models
6. Compare train/validation performance
7. Evaluate the selected model on the test set
8. Provide actionable recommendations

**Important:** This is an **image classification** notebook. It predicts whether an image belongs to `With Helmet` or `Without Helmet`. It does not draw bounding boxes around people or helmets. For real-time compliance monitoring with locations of helmets/persons, use object detection such as YOLO, SSD, or Faster R-CNN.

# Problem Statement

## Business Context

Workplace safety in hazardous environments such as construction sites and industrial plants is critical. One important safety requirement is ensuring workers wear safety helmets. Manual monitoring is not scalable, can be inconsistent, and is prone to human error.

SafeGuard Corp wants to automate safety helmet compliance checks using computer vision. The goal is to classify images into two categories:

- **With Helmet**: Workers wearing safety helmets
- **Without Helmet**: Workers not wearing safety helmets

## Objective

Build and compare CNN-based image classification models to identify whether a worker is wearing a helmet. The final model should prioritize reliable detection of **Without Helmet** cases, because missing a non-compliant worker creates a safety risk.

## Data Description

The dataset contains **631 images**:

- **With Helmet**: 311 images
- **Without Helmet**: 320 images

Images contain different lighting conditions, poses, backgrounds, and work environments.

# Review of the Monkey Species Notebook Approach

The Monkey Species notebook used a structured transfer-learning workflow:

- Images were resized to **64 × 64 × 3**.
- Labels were encoded for classification.
- Data was split into **train, validation, and test** sets.
- Inputs were normalized by dividing pixel values by 255.
- A pre-trained **VGG16** model with `include_top=False` was used as the feature extractor.
- VGG16 layers were frozen first.
- Multiple models were compared:
  - VGG16 base classifier
  - VGG16 base + feed-forward neural network
  - VGG16 base + feed-forward neural network + data augmentation
- Models were evaluated with accuracy, recall, precision, F1-score, and confusion matrix.
- Final model selection was based on validation/test performance and overfitting behavior.

This helmet notebook follows the same pattern and adds a simple CNN baseline for comparison.

# Installing and Importing the Necessary Libraries

Run the installation cell only if needed. If you are using Google Colab, restart the runtime after installation and then run all cells again from the imports cell.

In [ ]:
# Run only if needed
# !pip install numpy==1.25.2 pandas==2.0.3 seaborn==0.13.1 tensorflow==2.15.0 scikit-learn==1.2.2 matplotlib==3.7.1 opencv-python -q

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Reproducibility
SEED = 812
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Uncomment if full determinism is required. This can slow down GPU training.
# tf.config.experimental.enable_op_determinism()

# Data Overview

## Loading the Data

Set `DATA_DIR` to the folder that contains the two image-class subfolders. Common examples are:

```text
helmet_dataset/
    With Helmet/
    Without Helmet/
```

or

```text
helmet_dataset/
    with_helmet/
    without_helmet/
```

The loader below is intentionally flexible and detects image folders automatically.

In [ ]:
# Change this path to your dataset folder.
# In Google Colab, upload/extract the dataset and point DATA_DIR to that folder.
DATA_DIR = Path("helmet_dataset")

# Image settings follow the Monkey notebook pattern: 64 x 64 x 3
IMG_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Class mapping. We map Without Helmet to 1 because it is the higher-risk positive class.
CLASS_NAME_MAP = {
    "with helmet": 0,
    "with_helmet": 0,
    "helmet": 0,
    "yes": 0,
    "without helmet": 1,
    "without_helmet": 1,
    "no helmet": 1,
    "no_helmet": 1,
    "no": 1,
}
LABEL_NAMES = {0: "With Helmet", 1: "Without Helmet"}


def infer_label_from_folder(folder_name):
    name = folder_name.lower().replace("-", "_").strip()
    pretty_name = name.replace("_", " ")
    if name in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[name]
    if pretty_name in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[pretty_name]
    if "without" in name or "no_helmet" in name or "no helmet" in pretty_name:
        return 1
    if "with" in name or "helmet" in name:
        return 0
    return None


def load_images_from_folders(data_dir, img_size=64):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"DATA_DIR does not exist: {data_dir.resolve()}\n"
            "Update DATA_DIR to the folder containing the helmet image subfolders."
        )

    image_paths = [p for p in data_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    if len(image_paths) == 0:
        raise ValueError(f"No image files found under {data_dir.resolve()}.")

    images, labels, paths = [], [], []
    skipped = []

    for img_path in image_paths:
        label = None
        for parent in [img_path.parent] + list(img_path.parents):
            if parent == data_dir.parent:
                break
            label = infer_label_from_folder(parent.name)
            if label is not None:
                break
        if label is None:
            skipped.append(str(img_path))
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            skipped.append(str(img_path))
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)

        images.append(img)
        labels.append(label)
        paths.append(str(img_path))

    if len(images) == 0:
        raise ValueError(
            "Images were found, but labels could not be inferred from folder names. "
            "Use folders such as 'With Helmet' and 'Without Helmet'."
        )

    return np.array(images, dtype=np.uint8), pd.Series(labels, name="label"), paths, skipped

images, labels, image_paths, skipped_files = load_images_from_folders(DATA_DIR, IMG_SIZE)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Skipped files:", len(skipped_files))
print(labels.map(LABEL_NAMES).value_counts())

# Exploratory Data Analysis

## Plot Random Images From Each Class

In [ ]:
def show_random_images(images, labels, n_per_class=5):
    plt.figure(figsize=(15, 6))
    plot_idx = 1
    for class_id, class_name in LABEL_NAMES.items():
        idxs = np.where(labels.values == class_id)[0]
        sample_size = min(n_per_class, len(idxs))
        sampled = np.random.choice(idxs, sample_size, replace=False)
        for idx in sampled:
            plt.subplot(len(LABEL_NAMES), n_per_class, plot_idx)
            plt.imshow(images[idx])
            plt.title(class_name)
            plt.axis("off")
            plot_idx += 1
    plt.tight_layout()
    plt.show()

show_random_images(images, labels, n_per_class=5)

## Checking for Class Imbalance

In [ ]:
class_counts = labels.map(LABEL_NAMES).value_counts().rename_axis("Class").reset_index(name="Count")
class_counts["Percentage"] = round(class_counts["Count"] / class_counts["Count"].sum() * 100, 2)
display(class_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x=labels.map(LABEL_NAMES))
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.show()

# Data Preprocessing

## Important Note on Grayscale Conversion

The original helmet notebook had a section for grayscale conversion. For this transfer-learning workflow, we will **not convert images to grayscale**, because VGG16 trained on ImageNet expects **3-channel RGB images**. Keeping images in RGB makes the helmet dataset compatible with the Monkey notebook's VGG16 approach.

## Splitting the Dataset

We will use a stratified split so that both classes are represented proportionally in train, validation, and test sets.

In [ ]:
# First split: train+validation vs test
X_temp, X_test, y_temp, y_test = train_test_split(
    images,
    labels,
    test_size=0.15,
    random_state=SEED,
    stratify=labels
)

# Second split: train vs validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,  # 0.1765 of 85% ≈ 15%, giving about 70/15/15 total split
    random_state=SEED,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTrain class distribution:\n", y_train.map(LABEL_NAMES).value_counts(normalize=True).round(3))
print("\nValidation class distribution:\n", y_val.map(LABEL_NAMES).value_counts(normalize=True).round(3))
print("\nTest class distribution:\n", y_test.map(LABEL_NAMES).value_counts(normalize=True).round(3))

## Data Normalization

For the custom CNN, we normalize pixel values to the range `[0, 1]`.

For VGG16, the official ImageNet preprocessing is usually preferred. This notebook uses official VGG16 preprocessing for VGG models and `[0,1]` normalization for the custom CNN.

In [ ]:
# For custom CNN
X_train_normalized = X_train.astype("float32") / 255.0
X_val_normalized = X_val.astype("float32") / 255.0
X_test_normalized = X_test.astype("float32") / 255.0

# For VGG16 transfer learning
X_train_vgg = preprocess_input(X_train.astype("float32"))
X_val_vgg = preprocess_input(X_val.astype("float32"))
X_test_vgg = preprocess_input(X_test.astype("float32"))

# Labels as arrays
y_train_array = y_train.values.astype("int32")
y_val_array = y_val.values.astype("int32")
y_test_array = y_test.values.astype("int32")

# Model Building

## Model Evaluation Criterion

Because this is a safety-compliance problem, **recall for the Without Helmet class** is especially important. A false negative means the system predicts a worker is compliant when they are actually not wearing a helmet.

We will compare models using accuracy, weighted recall, weighted precision, weighted F1-score, `Without Helmet` recall, and confusion matrices.

## Utility Functions

In [ ]:
def get_predictions(model, predictors, threshold=0.5):
    # Return class predictions for binary sigmoid or 2-class softmax outputs.
    probs = model.predict(predictors, verbose=0)
    probs = np.asarray(probs)
    if probs.ndim == 2 and probs.shape[1] == 2:
        return probs.argmax(axis=1)
    return (probs.reshape(-1) >= threshold).astype(int)


def model_performance_classification(model, predictors, target, threshold=0.5):
    # Compute classification metrics.
    pred = get_predictions(model, predictors, threshold=threshold)
    target = np.asarray(target).reshape(-1)

    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred, average="weighted", zero_division=0)
    precision = precision_score(target, pred, average="weighted", zero_division=0)
    f1 = f1_score(target, pred, average="weighted", zero_division=0)
    without_helmet_recall = recall_score(target, pred, pos_label=1, zero_division=0)

    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1 Score": f1,
         "Without Helmet Recall": without_helmet_recall},
        index=[0],
    )
    return df_perf


def plot_confusion_matrix(model, predictors, target, threshold=0.5, title="Confusion Matrix"):
    pred = get_predictions(model, predictors, threshold=threshold)
    target = np.asarray(target).reshape(-1)
    cm = confusion_matrix(target, pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]],
                yticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()


def print_classification_report(model, predictors, target, threshold=0.5):
    pred = get_predictions(model, predictors, threshold=threshold)
    print(classification_report(np.asarray(target).reshape(-1), pred,
                                target_names=[LABEL_NAMES[0], LABEL_NAMES[1]], zero_division=0))


def plot_training_history(history, title):
    history_df = pd.DataFrame(history.history)
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["loss"], label="Train Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
    if "accuracy" in history_df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(history_df["accuracy"], label="Train Accuracy")
        plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
        plt.title(title + " - Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.show()


def visualize_predictions(model, predictors_for_model, original_images, target, n=8, threshold=0.5):
    pred = get_predictions(model, predictors_for_model, threshold=threshold)
    target = np.asarray(target).reshape(-1)
    idxs = np.random.choice(np.arange(len(original_images)), size=min(n, len(original_images)), replace=False)
    plt.figure(figsize=(16, 6))
    for plot_i, idx in enumerate(idxs, start=1):
        plt.subplot(2, 4, plot_i)
        plt.imshow(original_images[idx])
        color = "green" if pred[idx] == target[idx] else "red"
        plt.title(f"Actual: {LABEL_NAMES[target[idx]]}\nPred: {LABEL_NAMES[pred[idx]]}", color=color, fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## Training Setup

In [ ]:
epochs = 20
batch_size = 32

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),
]

## Model 1: Simple Convolutional Neural Network (CNN)

This is a baseline CNN trained from scratch. It helps us see whether transfer learning adds value.

In [ ]:
model_1 = Sequential(name="Simple_CNN")
model_1.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 3)))
model_1.add(BatchNormalization())
model_1.add(MaxPooling2D((2, 2)))
model_1.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
model_1.add(BatchNormalization())
model_1.add(MaxPooling2D((2, 2)))
model_1.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
model_1.add(BatchNormalization())
model_1.add(MaxPooling2D((2, 2)))
model_1.add(Flatten())
model_1.add(Dense(128, activation="relu"))
model_1.add(Dropout(0.4))
model_1.add(Dense(1, activation="sigmoid"))
model_1.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model_1.summary()

In [ ]:
history_model_1 = model_1.fit(
    X_train_normalized, y_train_array,
    epochs=epochs, batch_size=batch_size,
    validation_data=(X_val_normalized, y_val_array),
    callbacks=callbacks, verbose=1)
plot_training_history(history_model_1, "Model 1: Simple CNN")

In [ ]:
model_1_train_perf = model_performance_classification(model_1, X_train_normalized, y_train_array)
model_1_valid_perf = model_performance_classification(model_1, X_val_normalized, y_val_array)
print("Train performance metrics")
display(model_1_train_perf)
print("Validation performance metrics")
display(model_1_valid_perf)
plot_confusion_matrix(model_1, X_val_normalized, y_val_array, title="Model 1 Validation Confusion Matrix")
print_classification_report(model_1, X_val_normalized, y_val_array)

### Visualizing Model 1 Predictions

In [ ]:
visualize_predictions(model_1, X_val_normalized, X_val, y_val_array, n=8)

## Model 2: VGG16 Base

This follows the Monkey notebook's first transfer-learning model: VGG16 with `include_top=False`, frozen convolutional layers, flattening, and a classifier head.

In [ ]:
vgg_base_2 = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
for layer in vgg_base_2.layers:
    layer.trainable = False
model_2 = Sequential(name="VGG16_Base")
model_2.add(vgg_base_2)
model_2.add(Flatten())
model_2.add(Dense(1, activation="sigmoid"))
model_2.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model_2.summary()

In [ ]:
train_datagen_basic = ImageDataGenerator()
history_model_2 = model_2.fit(
    train_datagen_basic.flow(X_train_vgg, y_train_array, batch_size=batch_size, seed=SEED, shuffle=True),
    epochs=epochs,
    steps_per_epoch=max(1, X_train_vgg.shape[0] // batch_size),
    validation_data=(X_val_vgg, y_val_array),
    callbacks=callbacks,
    verbose=1)
plot_training_history(history_model_2, "Model 2: VGG16 Base")

In [ ]:
model_2_train_perf = model_performance_classification(model_2, X_train_vgg, y_train_array)
model_2_valid_perf = model_performance_classification(model_2, X_val_vgg, y_val_array)
print("Train performance metrics")
display(model_2_train_perf)
print("Validation performance metrics")
display(model_2_valid_perf)
plot_confusion_matrix(model_2, X_val_vgg, y_val_array, title="Model 2 Validation Confusion Matrix")
print_classification_report(model_2, X_val_vgg, y_val_array)

### Visualizing Model 2 Predictions

In [ ]:
visualize_predictions(model_2, X_val_vgg, X_val, y_val_array, n=8)

## Model 3: VGG16 Base + Feed-Forward Neural Network

This follows the Monkey notebook's second VGG16 model by adding dense layers and dropout after the frozen VGG16 feature extractor.

In [ ]:
vgg_base_3 = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
for layer in vgg_base_3.layers:
    layer.trainable = False
model_3 = Sequential(name="VGG16_Base_FFNN")
model_3.add(vgg_base_3)
model_3.add(Flatten())
model_3.add(Dense(256, activation="relu"))
model_3.add(Dropout(0.4))
model_3.add(Dense(32, activation="relu"))
model_3.add(Dense(1, activation="sigmoid"))
model_3.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model_3.summary()

In [ ]:
history_model_3 = model_3.fit(
    train_datagen_basic.flow(X_train_vgg, y_train_array, batch_size=batch_size, seed=SEED, shuffle=True),
    epochs=epochs,
    steps_per_epoch=max(1, X_train_vgg.shape[0] // batch_size),
    validation_data=(X_val_vgg, y_val_array),
    callbacks=callbacks,
    verbose=1)
plot_training_history(history_model_3, "Model 3: VGG16 Base + FFNN")

In [ ]:
model_3_train_perf = model_performance_classification(model_3, X_train_vgg, y_train_array)
model_3_valid_perf = model_performance_classification(model_3, X_val_vgg, y_val_array)
print("Train performance metrics")
display(model_3_train_perf)
print("Validation performance metrics")
display(model_3_valid_perf)
plot_confusion_matrix(model_3, X_val_vgg, y_val_array, title="Model 3 Validation Confusion Matrix")
print_classification_report(model_3, X_val_vgg, y_val_array)

### Visualizing Model 3 Predictions

In [ ]:
visualize_predictions(model_3, X_val_vgg, X_val, y_val_array, n=8)

## Model 4: VGG16 Base + FFNN + Data Augmentation

This follows the Monkey notebook's third VGG16 model. Data augmentation helps reduce overfitting, which is especially useful because this dataset has only 631 images. Augmentation is applied only to the training data.

In [ ]:
vgg_base_4 = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
for layer in vgg_base_4.layers:
    layer.trainable = False
model_4 = Sequential(name="VGG16_Base_FFNN_DataAug")
model_4.add(vgg_base_4)
model_4.add(Flatten())
model_4.add(Dense(256, activation="relu"))
model_4.add(Dropout(0.4))
model_4.add(Dense(32, activation="relu"))
model_4.add(Dense(1, activation="sigmoid"))
model_4.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model_4.summary()

In [ ]:
train_datagen_augmented = ImageDataGenerator(
    rotation_range=20,
    fill_mode="nearest",
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True)

history_model_4 = model_4.fit(
    train_datagen_augmented.flow(X_train_vgg, y_train_array, batch_size=batch_size, seed=SEED, shuffle=True),
    epochs=epochs,
    steps_per_epoch=max(1, X_train_vgg.shape[0] // batch_size),
    validation_data=(X_val_vgg, y_val_array),
    callbacks=callbacks,
    verbose=1)
plot_training_history(history_model_4, "Model 4: VGG16 Base + FFNN + Data Augmentation")

In [ ]:
model_4_train_perf = model_performance_classification(model_4, X_train_vgg, y_train_array)
model_4_valid_perf = model_performance_classification(model_4, X_val_vgg, y_val_array)
print("Train performance metrics")
display(model_4_train_perf)
print("Validation performance metrics")
display(model_4_valid_perf)
plot_confusion_matrix(model_4, X_val_vgg, y_val_array, title="Model 4 Validation Confusion Matrix")
print_classification_report(model_4, X_val_vgg, y_val_array)

### Visualizing Model 4 Predictions

In [ ]:
visualize_predictions(model_4, X_val_vgg, X_val, y_val_array, n=8)

# Model Performance Comparison and Final Model Selection

The final model should have strong validation performance and should not show a large train-validation gap. For this business problem, `Without Helmet Recall` is very important.

In [ ]:
models_train_comp_df = pd.concat([
    model_1_train_perf.T, model_2_train_perf.T, model_3_train_perf.T, model_4_train_perf.T], axis=1)
models_train_comp_df.columns = ["Simple CNN", "VGG16 Base", "VGG16 Base + FFNN", "VGG16 Base + FFNN + Data Aug"]

models_valid_comp_df = pd.concat([
    model_1_valid_perf.T, model_2_valid_perf.T, model_3_valid_perf.T, model_4_valid_perf.T], axis=1)
models_valid_comp_df.columns = models_train_comp_df.columns

print("Training Performance")
display(models_train_comp_df)
print("Validation Performance")
display(models_valid_comp_df)
print("Train - Validation Gap")
display(models_train_comp_df - models_valid_comp_df)

In [ ]:
# Select the best model using validation F1 first, then Without Helmet Recall as a tie-breaker.
valid_summary = models_valid_comp_df.T.reset_index().rename(columns={"index": "Model"})
valid_summary = valid_summary.sort_values(by=["F1 Score", "Without Helmet Recall", "Accuracy"], ascending=False)
display(valid_summary)

best_model_name = valid_summary.iloc[0]["Model"]
print("Best model based on validation F1, safety recall, and accuracy:", best_model_name)

model_lookup = {
    "Simple CNN": (model_1, X_test_normalized),
    "VGG16 Base": (model_2, X_test_vgg),
    "VGG16 Base + FFNN": (model_3, X_test_vgg),
    "VGG16 Base + FFNN + Data Aug": (model_4, X_test_vgg),
}
final_model, X_test_for_final = model_lookup[best_model_name]

## Test Performance of the Selected Final Model

In [ ]:
final_test_perf = model_performance_classification(final_model, X_test_for_final, y_test_array)
print("Selected model:", best_model_name)
print("Test performance metrics")
display(final_test_perf)
plot_confusion_matrix(final_model, X_test_for_final, y_test_array, title=f"{best_model_name} Test Confusion Matrix")
print_classification_report(final_model, X_test_for_final, y_test_array)
visualize_predictions(final_model, X_test_for_final, X_test, y_test_array, n=8)

# Final Recommendation

Use the model selected by the validation comparison cell above. In most runs on a small dataset like this, the recommended model is expected to be:

## **VGG16 Base + FFNN + Data Augmentation**

### Why this is the strongest default recommendation

- The dataset is small, so training a CNN from scratch can overfit.
- VGG16 provides strong pre-trained image features learned from ImageNet.
- The FFNN head adapts VGG16 features to the helmet/no-helmet business problem.
- Data augmentation improves generalization by exposing the model to shifted, rotated, and zoomed images.
- For safety use cases, choose the model with the best validation/test balance and high **Without Helmet Recall**.

### Business Recommendation

- Deploy the final model only after checking test-set performance and a confusion matrix.
- Prioritize reducing **false negatives** for `Without Helmet`, because missing a safety violation is more costly than flagging an image for manual review.
- Add a manual review queue for low-confidence predictions.
- Collect more real site images with different camera angles, lighting, helmet colors, occlusions, and worker distances.
- For real-time monitoring where the system must locate each person and helmet, upgrade from image classification to object detection using YOLO, SSD, or Faster R-CNN.

### Technical Recommendations

- Keep VGG16 frozen initially, then optionally fine-tune the last VGG block with a very low learning rate.
- Try larger image sizes such as 128×128 or 224×224 if compute allows, because helmets can be small in the image.
- Track recall for `Without Helmet`, precision, F1-score, and confusion matrix in every experiment.
- Save the selected model with `model.save()` for deployment.

# Optional: Fine-Tuning the Best VGG16 Model

Run this only after the frozen VGG16 model is stable. Fine-tuning may improve performance, but with a small dataset it can also overfit.

In [ ]:
# Optional fine-tuning example for Model 4.
# Unfreeze the last convolutional block and train with a very low learning rate.

# for layer in vgg_base_4.layers:
#     layer.trainable = False
#
# for layer in vgg_base_4.layers:
#     if layer.name.startswith("block5"):
#         layer.trainable = True
#
# model_4.compile(optimizer=Adam(learning_rate=1e-5), loss="binary_crossentropy", metrics=["accuracy"])
#
# history_model_4_finetuned = model_4.fit(
#     train_datagen_augmented.flow(X_train_vgg, y_train_array, batch_size=batch_size, seed=SEED, shuffle=True),
#     epochs=10,
#     steps_per_epoch=max(1, X_train_vgg.shape[0] // batch_size),
#     validation_data=(X_val_vgg, y_val_array),
#     callbacks=callbacks,
#     verbose=1,
# )
#
# model_4_finetuned_valid_perf = model_performance_classification(model_4, X_val_vgg, y_val_array)
# display(model_4_finetuned_valid_perf)

# Save the Final Model

In [ ]:
# Save selected final model
# final_model.save("helmet_final_model.keras")
# print("Saved model to helmet_final_model.keras")

<font size=5 color='blue'>Power Ahead!</font>
___